In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

import pandas as pd
import scanpy as sc

import dianne
dianne.setNotebookWidth(100)

In [21]:
# Load the data and prepare the patches for annotation and training the classifier
outsSTQpath = '/projects/activities/kappsen-tmc/USERS/domans/results-STQ-post-xenium-32-final/'
samples  = ['JDC-WP-012-w']

classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)

df_ids = pd.read_csv('/projects/activities/kappsen-tmc/USERS/domans/Pancreas Xenium identifiers and paths.csv', index_col=0)
samplesToSTQnames = df_ids['HE-slide-identifier'].to_dict()

ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L = dianne.loadDataAndPreparePatchesStatic(samples, outsSTQpath, fname='img.data.ctranspath-1.h5ad',
                                                                                            L=None, ts=56, mpp=0.25, N=8, samplesToSTQnames=samplesToSTQnames)

In [22]:
xeniumBundlePaths = df_ids['Xenium-CODEX-outs'].dropna().str.replace('/xe/', '/xe/spaceranger-custom-seg-outs/').to_dict()
xeniumMatrixPaths = df_ids['HE-matrix'].to_dict()

cell_barcodes = {s: pd.read_csv(xeniumBundlePaths[s] + '/analysis/clustering/gene_expression_kmeans_5_clusters/clusters.csv')['Barcode'].astype(str) for s in samples}
cell_barcodes_inv = {s: {v: k for k, v in cell_barcodes[s].items()} for s in samples}

annotationsd = {s: (pd.read_csv(xeniumBundlePaths[s] + '/analysis/clustering/gene_expression_kmeans_5_clusters/clusters.csv')['Cluster'].astype(str) + '').astype('category') for s in samples}

p = '/sdata/activities/kappsen-tmc/visium/analysis/downstream/pancreas-Xenium-pre-processed-32'
annotations = {}
for sample in samples:
    se = sc.read_h5ad(p + f'/xenium-{sample}-v2-processed.h5ad').obs['Celltype fine']
    se.index = se.index.str.split('.').str[0]
    se.index = [cell_barcodes_inv[sample][i] if i in cell_barcodes_inv[sample] else i for i in se.index]
    annotations[sample] = se

In [ ]:
# Display representative patches that can be used to start the annotation process
startParams = {}
dianne.jumpStart(patchCoordinates, patchesCDFs, imgs, startParams, ncols=6, nrows=5, ads=ads,
                 L=1 if L is None else L, sh=int(ts/2)/mpp, figsize=(3, 3), seed=1, maxN=10**3)

In [37]:
# Run the annotation loop with active learning and probabilistic patch selection
clf, plog, bp = {}, [], {}
try: bp.update({startParams['selected_patch']: 'positive'})
except: pass
dianne.runAnnotation(patchCoordinates, patchesCDFs, imgs, bp, clf, plog, ads=ads, qs=qs, minN=1,
                    figsize=(5, 5), augFunc=dianne.PCMA, alpha=0.5, R=1, cmapColors=['lightcoral', 'blue'], # (5, 5)
                    seed=0, randomness=0.5, L=1 if L is None else L, sh=int(ts/2)/mpp,
                    xeniumBundlePaths=xeniumBundlePaths, xeniumMatrixPaths=xeniumMatrixPaths, numColumns=1,
                    transcriptsAlpha=0.05, transcriptsColor='lime', transcriptsSize=2., loadTranscripts=False,
                    selectedGenes=['MUC5AC'], minCount=1, loadCells=True, loadCellBoundaries=True,
                    showAnnotations=False, annotations=annotations, annotationsPalette=dianne.Set123)

In [28]:
# To run inference on the entire slides
infSample = samples[0]
x, y, p = dianne.inferProb(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=2)
dianne.showProbImg(x, y, p, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample)